In [22]:
%pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [23]:
# Making sure all columns show
import pandas as pd, numpy as np
df = pd.read_csv("adult.csv")
print(df.shape)
df.info()
df.describe(include='all')

(32561, 15)
<class 'pandas.DataFrame'>
RangeIndex: 32561 entries, 0 to 32560
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   age             32561 non-null  int64
 1   workclass       32561 non-null  str  
 2   fnlwgt          32561 non-null  int64
 3   education       32561 non-null  str  
 4   education.num   32561 non-null  int64
 5   marital.status  32561 non-null  str  
 6   occupation      32561 non-null  str  
 7   relationship    32561 non-null  str  
 8   race            32561 non-null  str  
 9   sex             32561 non-null  str  
 10  capital.gain    32561 non-null  int64
 11  capital.loss    32561 non-null  int64
 12  hours.per.week  32561 non-null  int64
 13  native.country  32561 non-null  str  
 14  income          32561 non-null  str  
dtypes: int64(6), str(9)
memory usage: 3.7 MB


,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
count,32561.000000,32561,3.256100e+04,32561,32561.000000,32561,32561,32561,32561,32561,32561.000000,32561.000000,32561.000000,32561,32561
unique,NaN,9,NaN,16,NaN,7,15,6,5,2,NaN,NaN,NaN,42,2
top,NaN,Private,NaN,HS-grad,NaN,Married-civ-spouse,Prof-specialty,Husband,White,Male,NaN,NaN,NaN,United-States,<=50K
freq,NaN,22696,NaN,10501,NaN,14976,4140,13193,27816,21790,NaN,NaN,NaN,29170,24720
mean,38.581647,NaN,1.897784e+05,NaN,10.080679,NaN,NaN,NaN,NaN,NaN,1077.648844,87.303830,40.437456,NaN,NaN
std,13.640433,NaN,1.055500e+05,NaN,2.572720,NaN,NaN,NaN,NaN,NaN,7385.292085,402.960219,12.347429,NaN,NaN
min,17.000000,NaN,1.228500e+04,NaN,1.000000,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,1.000000,NaN,NaN
25%,28.000000,NaN,1.178270e+05,NaN,9.000000,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,40.000000,NaN,NaN
50%,37.000000,NaN,1.783560e+05,NaN,10.000000,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,40.000000,NaN,NaN
75%,48.000000,NaN,2.370510e+05,NaN,12.000000,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,45.000000,NaN,NaN


In [24]:
# For some reason this dataset has missing values as ?, so we're replacing them with NAN.
df = df.replace("?", np.nan)
# strip whitespace from all text columns
for c in df.select_dtypes(include='object').columns:
    df[c] = df[c].str.strip()

C:\Users\louis\AppData\Local\Temp\ipykernel_40376\4030268877.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for c in df.select_dtypes(include='object').columns:


In [25]:
# Drop all missing values
df = df.dropna()

In [26]:
# Remove duplicates
df = df.drop_duplicates()

In [ ]:
# Dropping Education since education.num is essentially an encoded numeric representation that's easier to work with. Here is the matching:
'''
Preschool	1
1st-4th	2
5th-6th	3
7th-8th	4
9th	5
10th	6
11th	7
12th	8
HS-grad	9
Some-college	10
Assoc-voc	11
Assoc-acdm	12
Bachelors	13
Masters	14
Prof-school	15
Doctorate	16
'''
# Also dropping fnlwgt since its just a sampling weight
df = df.drop(columns=['education', 'fnlwgt'])

In [ ]:
# Encoding income to be binary, 0 if less than 50k, 1 if more.
df['income'] = df['income'].map({'<=50K': 0, '>50K': 1})

In [ ]:
# Turning native country into either US or Other, since its 90% U.S
df['native.country'] = np.where(df['native.country']=='United-States', 'US', 'Other')

In [ ]:
# Turning all the string values into binary values.
df = pd.get_dummies(df, columns=['workclass','marital.status','occupation',
                                 'relationship','race','sex','native.country'],
                    drop_first=True)

In [ ]:
# Split dataset for training into 80/20 data,
from sklearn.model_selection import train_test_split
X = df.drop(columns=['income'])
y = df['income']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
# Scale numeric features and fit the scaler on the training data
from sklearn.preprocessing import StandardScaler
num_cols = ['age','education.num','capital.gain','capital.loss','hours.per.week']
scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols]  = scaler.transform(X_test[num_cols])

In [ ]:
# Just making sure everything is good lol
print(X_train.isna().sum().sum(), "missing in train") 
print(X_train.dtypes.value_counts())
print(X_train.shape, X_test.shape)

0 missing in train
bool       36
float64     5
Name: count, dtype: int64
(24111, 41) (6028, 41)
